In [1]:
!pip install -q roboflow

from roboflow import Roboflow
import zipfile
import os
import shutil

# Download dataset
rf = Roboflow(api_key="1LXmrHIjxoP1M8QXA59l")
project = rf.workspace("urpc2020").project("urpc2020")
version = project.version(2)
dataset = version.download("yolov12")   # downloads to ./URPC2020-2

# Path to downloaded dataset folder
dataset_dir = "URPC2020-2"

# Create a ZIP file
zip_name = "urpc2020_dataset.zip"
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(dataset_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, start=os.path.dirname(dataset_dir))
            zipf.write(file_path, arcname)

print(f"✅ Dataset saved as {zip_name} (size: {os.path.getsize(zip_name) / (1024*1024):.2f} MB)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 5.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 35.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 113.8 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to URPC2020-2 in yolov12:: 100%|██████████| 37894/37894 [00:05<00:00, 7182.20it/s] 


✅ Dataset saved as urpc2020_dataset.zip (size: 1085.14 MB)


In [2]:
# %% [markdown]
# # YOLO11 + Cross‑Scale Efficient Attention (CSEA) – URPC2020
# ## Novelty: Fast cross‑scale mixing + shared channel attention + spatial gate
# ## Fixed & Fully Runnable

# ------------------------------
# 1. Install
# ------------------------------
!pip install -q ultralytics matplotlib seaborn

# ------------------------------
# 2. Imports & Dataset Setup
# ------------------------------
import os
import yaml
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics import YOLO
from ultralytics.nn.modules import C2f

# ------------------------------
# 2b. Choose model size here
# ------------------------------
model_size = 'm'          # Options: 'n', 's', 'm', 'l'
model_name = f"yolo11{model_size}.pt"
print(f"Using model: {model_name}")

# Adjusted batch sizes to avoid OOM
batch_sizes = {'n': 16, 's': 10, 'm': 6, 'l': 4}
batch = batch_sizes.get(model_size, 10)

# Dataset Path (autodetect for URPC2020 on Kaggle)
dataset_dir = "/kaggle/working/URPC2020-2"
data_yaml = os.path.join(dataset_dir, "data.yaml")

if not os.path.exists(data_yaml):
    for path in ["/kaggle/input/urpc2020/data.yaml", "/kaggle/input/urpc2020-2/data.yaml"]:
        if os.path.exists(path):
            data_yaml = path
            dataset_dir = os.path.dirname(path)
            break

# If still not found, you can manually set the path here
if not os.path.exists(data_yaml):
    raise FileNotFoundError("URPC2020 data.yaml not found. Please check dataset path.")

with open(data_yaml, 'r') as f:
    data_cfg = yaml.safe_load(f)

print(f"✅ Classes: {data_cfg['names']} ({data_cfg['nc']} classes)")
print(f"Dataset: {dataset_dir}")

# ------------------------------
# 3. NOVEL FUSION BLOCK: Cross‑Scale Efficient Attention (CSEA)
# ------------------------------
class CrossScaleEfficientAttention(nn.Module):
    """
    Novel: Cross‑scale mixing + shared channel attention + light spatial gate.
    Fully fixed (no Conv1d issues). Fast and low‑parameter.
    """
    def __init__(self, channels, reduction=4):
        super().__init__()
        # Depthwise convs for cross‑scale feature extraction (lightweight)
        self.conv3 = nn.Conv2d(channels, channels, 3, padding=1, groups=channels)
        self.conv5 = nn.Conv2d(channels, channels, 5, padding=2, groups=channels)
        
        # Channel attention (using 1x1 Conv2d – safe and efficient)
        self.channel_att = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, max(channels // reduction, 1), 1),
            nn.ReLU(),
            nn.Conv2d(max(channels // reduction, 1), channels, 1),
            nn.Sigmoid()
        )
        
        # Spatial gate (simple 1x1 conv)
        self.spatial_gate = nn.Conv2d(channels, 1, 1)
        self.act = nn.SiLU()
        
    def forward(self, x):
        # 1. Cross‑scale mixing
        x3 = self.conv3(x)
        x5 = self.conv5(x)
        fused = x + x3 + x5   # efficient multi‑scale fusion
        
        # 2. Channel attention
        ca_weight = self.channel_att(fused)
        ca_out = fused * ca_weight
        
        # 3. Spatial gate
        sg = torch.sigmoid(self.spatial_gate(ca_out))
        out = ca_out * sg
        
        return self.act(out)


class C2f_CSEA(C2f):
    """C2f with Cross‑Scale Efficient Attention (replaces original C2f)"""
    def __init__(self, c1, c2, n=1, shortcut=False, g=1, e=0.5):
        super().__init__(c1, c2, n, shortcut, g, e)
        self.attention = CrossScaleEfficientAttention(c2)

    def forward(self, x):
        # Standard C2f forward
        y = list(self.cv1(x).chunk(2, 1))
        y.extend(m(y[-1]) for m in self.m)
        out = self.cv2(torch.cat(y, 1))
        # Apply novel attention
        return self.attention(out)

# ------------------------------
# 4. Build Model and Inject Attention Blocks
# ------------------------------
model = YOLO(model_name)
torch_model = model.model.model

def replace_neck_with_csea(torch_model):
    # Find the SPPF layer (neck starts after it)
    sppf_idx = None
    for i, layer in enumerate(torch_model):
        if 'SPPF' in str(layer.__class__.__name__):
            sppf_idx = i
            break
    
    if sppf_idx is None:
        print("⚠️ SPPF not found – check model structure")
        return
    
    replaced = 0
    for i in range(sppf_idx + 1, len(torch_model)):
        layer = torch_model[i]
        if layer.__class__.__name__ in ['C2f', 'C3k2']:
            # Get output channels
            if hasattr(layer, 'cv2') and hasattr(layer.cv2, 'conv'):
                c_out = layer.cv2.conv.out_channels
            else:
                c_out = 256   # fallback
            
            # Replace with our C2f_CSEA
            new_layer = C2f_CSEA(c_out, c_out, n=1)  # n=1 is safe; only 1 bottleneck to avoid mismatch
            
            # Copy weights where possible (avoid training from scratch)
            with torch.no_grad():
                try:
                    new_layer.cv1.load_state_dict(layer.cv1.state_dict())
                    new_layer.cv2.load_state_dict(layer.cv2.state_dict())
                    # Copy the first bottleneck if it exists
                    if hasattr(layer, 'm') and len(layer.m) > 0 and hasattr(new_layer, 'm') and len(new_layer.m) > 0:
                        new_layer.m[0].load_state_dict(layer.m[0].state_dict())
                except Exception as e:
                    print(f"  (weight copy warning: {e})")
            
            torch_model[i] = new_layer
            replaced += 1
            print(f"✓ Replaced layer {i} with C2f_CSEA (cross‑scale efficient attention)")
    
    print(f"✅ Injected {replaced} novel attention blocks into YOLO11{model_size}")

replace_neck_with_csea(torch_model)

# ------------------------------
# 5. Training on 100% of URPC2020 Dataset (30 epochs)
# ------------------------------
results = model.train(
    data=data_yaml,
    epochs=30,
    imgsz=512,
    batch=batch,
    fraction=1.0,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=4,
    project='urpc_yolo11_csea',
    name=f'full_{model_size}_cross_scale_efficient_attention_100pct',
    exist_ok=True,
    optimizer='AdamW',
    lr0=0.001,
    augment=True,
    weight_decay=0.0005,
    patience=15
)

# ------------------------------
# 6. Evaluation
# ------------------------------
metrics = model.val(data=data_yaml, split='val', iou=0.5, conf=0.25)

print("\n" + "="*60)
print(f"FINAL EVALUATION - YOLO11{model_size} + Cross‑Scale Efficient Attention (100% Dataset, 30 epochs)")
print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"Precision    : {metrics.box.mp:.4f}")
print(f"Recall       : {metrics.box.mr:.4f}")
f1 = 2 * metrics.box.mp * metrics.box.mr / (metrics.box.mp + metrics.box.mr + 1e-6)
print(f"F1-Score     : {f1:.4f}")
print("="*60)

# ------------------------------
# 7. Inference Examples on Raw Images
# ------------------------------
# Path to your trained best model (automatically used from training)
best_model_path = model.trainer.best if hasattr(model, 'trainer') and model.trainer else f"urpc_yolo11_csea/full_{model_size}_cross_scale_efficient_attention_100pct/weights/best.pt"

print(f"\n📸 Running inference on sample images (if available)...")

# Example: predict on a single raw image (adjust path as needed)
sample_image = "/kaggle/working/URPC2020-2/test/images/000001_jpg.rf.5dd5b95eabd2af019b59ef5104b07dc3.jpg"   # typical URPC filename
if os.path.exists(sample_image):
    results = model(sample_image, save=True, conf=0.25)
    print(f"✅ Prediction saved for {sample_image}")
else:
    print(f"⚠️ Sample image not found at {sample_image}. To predict: model('your_image.jpg', save=True)")

print(f"\n🎉 Done! YOLO11{model_size} + Cross‑Scale Efficient Attention trained and evaluated on 100% of URPC2020 (30 epochs).")
print(f"Trained model saved at: {best_model_path}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.9 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using model: yolo11m.pt
✅ Classes: ['echinus', 'holothurian', 'scallop', 'starfish'] (4 classes)
Dataset: /kaggle/working/URPC2020-2
  (weight copy warning: Error(s) in loading state_dict for Conv:
	size mismatch for conv.weight: copying a param with shape torch.Size([512, 1024, 1, 1]) from checkpoint, the shape in current model is torch.Size([512, 512, 1, 1]).)
✓ Replaced layer 13 with C2f_CSEA (cross‑scale efficient attention)
  (weight copy warning: Error(s) in loading state_dict for Conv:
	size mismatch for conv.weight: copying a param with shape torch.Size([256, 1024, 1, 1]) from checkpoint,

In [8]:
# ------------------------------
# 8. Save best model path and copy model file for later use
# ------------------------------
import shutil

# Save the path to a text file
path_txt = "/kaggle/working/best_model_path.txt"
with open(path_txt, "w") as f:
    f.write(best_model_path)
print(f"✅ Best model path saved to {path_txt}")

# Optional: copy the best model to a well-known output location (e.g., working directory)
output_model_path = "/kaggle/working/best_model.pt"
shutil.copy(best_model_path, output_model_path)
print(f"✅ Best model file copied to {output_model_path}")

print(f"\n📦 To download later:")
print(f"   - Path file: {path_txt}")
print(f"   - Model file: {output_model_path}")

✅ Best model path saved to /kaggle/working/best_model_path.txt
✅ Best model file copied to /kaggle/working/best_model.pt

📦 To download later:
   - Path file: /kaggle/working/best_model_path.txt
   - Model file: /kaggle/working/best_model.pt
